In [1]:
pip install pandas pillow albumentations cairosvg openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 3.1 MB/s eta 0:00:00


In [20]:
# Replace 'your_zip_file.zip' with the actual name of your uploaded zip file
# This command unzips the file into the current directory, which should be /content/
!unzip /content/svg_images.zip

Archive:  /content/svg_images.zip
replace ADAVNCEDAIPATT/pattern_10_vector.svg? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: ADAVNCEDAIPATT/pattern_10_vector.svg  
replace ADAVNCEDAIPATT/pattern_11_vector.svg? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: ADAVNCEDAIPATT/pattern_11_vector.svg  
replace ADAVNCEDAIPATT/pattern_12_vector.svg? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: ADAVNCEDAIPATT/pattern_12_vector.svg  
replace ADAVNCEDAIPATT/pattern_13_vector.svg? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: ADAVNCEDAIPATT/pattern_13_vector.svg  
replace ADAVNCEDAIPATT/pattern_14_vector.svg? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: ADAVNCEDAIPATT/pattern_14_vector.svg  
replace ADAVNCEDAIPATT/pattern_15_vector.svg? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: ADAVNCEDAIPATT/pattern_15_vector.svg  
replace ADAVNCEDAIPATT/pattern_16_vector.svg? [y]es, [n]o, [A]ll, [N]one, [r]ename: yy
  inflating: ADAVNCEDAIPATT/pattern_16_vector.svg  

In [28]:
import os
import random
import pandas as pd
import numpy as np
from PIL import Image, ImageEnhance
import albumentations as A
import cairosvg

# ---------------------------
# SETTINGS
# ---------------------------
INPUT_EXCEL = "Saudi_Heritage_Patterns_Catalog.xlsx"
SVG_FOLDER = "ADAVNCEDAIPATT" # Changed from "svg_images"
PNG_FOLDER = "converted_png"
OUTPUT_FOLDER = "augmented_images"
OUTPUT_EXCEL = "augmented_dataset.xlsx"

TOTAL_IMAGES = 100
IMG_SIZE = 512  # match your required size

os.makedirs(PNG_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# ---------------------------
# LOAD ORIGINAL DATA
# ---------------------------
df = pd.read_excel('/content/Data.xlsx', header=1)

# ---------------------------
# SVG → PNG CONVERSION
# ---------------------------
def convert_svg_to_png(svg_path, png_path):
    cairosvg.svg2png(
        url=svg_path,
        write_to=png_path,
        output_width=IMG_SIZE,
        output_height=IMG_SIZE
    )

# Convert all SVGs
for idx, row in df.iterrows():
    svg_file = os.path.join(SVG_FOLDER, row["Image Name"])
    png_file = os.path.join(PNG_FOLDER, row["Image Name"].replace(".svg", ".png"))

    if not os.path.exists(png_file):
        convert_svg_to_png(svg_file, png_file)

# ---------------------------
# AUGMENTATION PIPELINE
# ---------------------------
transform = A.Compose([
    A.Rotate(limit=15, p=0.7),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=0, p=0.5)
])

# ---------------------------
# GENERATE AUGMENTED DATA
# ---------------------------
new_rows = []

for i in range(TOTAL_IMAGES):

    # Pick random original image
    row = df.sample(1).iloc[0]

    img_name = row["Image Name"].replace(".svg", ".png") # Changed 'image Name' to 'Image Name'
    img_path = os.path.join(PNG_FOLDER, img_name)

    image = Image.open(img_path).convert("RGB")
    img_np = np.array(image)

    # Apply augmentation
    augmented = transform(image=img_np)['image']
    aug_img = Image.fromarray(augmented)

    # ---------------------------
    # DENSITY CONTROL (MAIN PART)
    # ---------------------------
    density = random.choice(["Sparse", "Medium", "Dense"])

    enhancer = ImageEnhance.Contrast(aug_img)

    if density == "Dense":
        aug_img = enhancer.enhance(1.3)
    elif density == "Sparse":
        aug_img = enhancer.enhance(0.7)

    # ---------------------------
    # SAVE IMAGE
    # ---------------------------
    new_filename = f"aug_{i+1}_{density}.png"
    save_path = os.path.join(OUTPUT_FOLDER, new_filename)
    aug_img.save(save_path)

    # ---------------------------
    # CREATE NEW EXCEL ROW
    # ---------------------------

    description = f"{row['Symmetry & Repetition'].split('/')[0].strip()} geometric pattern arranged in a structured layout. Features {row[' Motif Type']} with {row[' Color Palette']}. Background follows original design. Overall pattern shows {density.lower()} density."

    new_row = {
        "Image Name": new_filename,
        "Color Palette": row[" Color Palette"],
        "Motif Type": row[" Motif Type"],
        "Motif Density": row["Motif density"],
        "Complexity Level": row[" Complexity Level"],
        "Symmetry & Repetition": row["Symmetry & Repetition"],
        "Description": description,
        "Region": row["Region "]
    }

    new_rows.append(new_row)

# ---------------------------
# SAVE NEW EXCEL
# ---------------------------
new_df = pd.DataFrame(new_rows)
new_df.to_excel(OUTPUT_EXCEL, index=False)

print("\u2705 DONE!")
print(f"Generated {TOTAL_IMAGES} images")
print(f"Saved to: {OUTPUT_FOLDER}")
print(f"Excel file: {OUTPUT_EXCEL}")

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


✅ DONE!
Generated 100 images
Saved to: augmented_images
Excel file: augmented_dataset.xlsx


In [26]:
print(df.columns)

Index(['Image Name', ' Color Palette', ' Motif Type', 'Motif density',
       ' Complexity Level', 'Symmetry & Repetition', 'Description', 'Region '],
      dtype='object')


In [27]:
print(df.columns)

Index(['Image Name', ' Color Palette', ' Motif Type', 'Motif density',
       ' Complexity Level', 'Symmetry & Repetition', 'Description', 'Region '],
      dtype='object')


In [22]:
import os

excel_file_path = '/content/Data.xlsx'
svg_folder_path = '/content/svg_images'

print(f"Checking for '{excel_file_path}': {os.path.exists(excel_file_path)}")
print(f"Checking for '{svg_folder_path}': {os.path.exists(svg_folder_path) and os.path.isdir(svg_folder_path)}")

Checking for '/content/Data.xlsx': True
Checking for '/content/svg_images': False


In [25]:
import os

excel_file_path = '/content/Data.xlsx'
svg_folder_path = '/content/ADAVNCEDAIPATT' # Updated path

print(f"Checking for '{excel_file_path}': {os.path.exists(excel_file_path)}")
print(f"Checking for '{svg_folder_path}': {os.path.exists(svg_folder_path) and os.path.isdir(svg_folder_path)}")

Checking for '/content/Data.xlsx': True
Checking for '/content/ADAVNCEDAIPATT': True
